# NHD Feature Matcher
Match fracking water sources to NHD features using fuzzy name matching
constrained by proximity to well coordinates.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
from rapidfuzz import fuzz
from math import radians, cos, sin, asin, sqrt

print('Loading NHD...')
nhd_fl = gpd.read_file('data/NHD_PA_named.gpkg', layer='NHDFlowline')
nhd_wb = gpd.read_file('data/NHD_PA_named.gpkg', layer='NHDWaterbody')
nhd = pd.concat([nhd_fl, nhd_wb], ignore_index=True)
print(f'  Flowlines:   {len(nhd_fl):,}')
print(f'  Waterbodies: {len(nhd_wb):,}')
print(f'  Total NHD:   {len(nhd):,}')

In [ ]:
SKINNY_PATH = r'G:\My Drive\production\repos\openFF_data_2026_04_03\skinny_df.parquet'

jct    = pd.read_parquet('data/well_junction_table.parquet')
master = pd.read_parquet('data/FPW_master_water_source.parquet')

well_coords = (
    pd.read_parquet(SKINNY_PATH, columns=['api10', 'bgLatitude', 'bgLongitude'])
    .groupby('api10', as_index=False).first()
    .rename(columns={'bgLatitude': 'well_lat', 'bgLongitude': 'well_lon'})
)
jct = jct.merge(well_coords, on='api10', how='left')

RULES = [
    ('reuse',          r'recycl|reuse|re-use|flowback|produced water|rain\s*water'),
    ('interconnection',r'\bintc\b|\btap\b|\bvending\b|\bvault\b|\bhydrant\b'
                       r'|\bauthority\b|\bmunicipal\b|water\s+co[^u]|\bmeter\b'
                       r'|public\s+water|\bpwsid\b|water\s+system|water\s+service'
                       r'|water\s+company|interconnect'),
    ('groundwater',    r'\bwell\b|\bspring\b|\baquifer\b|ground\s*water'),
    ('impoundment',    r'impoundment|\bpit\b|frac\s+pond'),
    ('surface_direct', r'creek|river|\brun\b|stream|\blake\b|\bpond\b|reservoir'
                       r'|brook|\bbranch\b|\bfork\b|hollow|\bdam\b|hatchery'),
    ('srbc_only',      r'srbc\s+docket'),
    ('dont_know',     r'\bSWW\b|\bSPWA\b|\bSWPA\b|\bNKWA\b|\bMAWC\b|\bMANK\b'
                      r'|\bAqua\b|\bWI\b|\bbrine\b|quarry|limestone\s+mine'
                      r'|\bClermont\b|\bAWS\b'),
]
def classify(name):
    if pd.isna(name): return 'no_source'
    for label, pat in RULES:
        if re.search(pat, str(name), re.IGNORECASE): return label
    return 'ambiguous'

jct['source_type'] = jct['planSource'].apply(classify)
jct['has_srbc']    = jct['planSource'].str.contains(r'srbc\s+docket', case=False, na=False)

cands = jct[jct['source_type'].isin(['surface_direct','impoundment','srbc_only'])].copy()
valid_id = cands['site_ID'].notna() & ~cands['site_ID'].isin(['','ambiguous'])
candidates = pd.concat([
    cands[valid_id].drop_duplicates('site_ID')[['site_ID','planSource','site_clean','source_type','has_srbc']],
    cands[~valid_id].drop_duplicates('planSource')[['site_ID','planSource','site_clean','source_type','has_srbc']],
], ignore_index=True)

candidates = candidates.merge(master[['site_ID','site_name','Latitude','Longitude']], on='site_ID', how='left')
well_proxy = (cands.groupby('planSource')[['well_lat','well_lon']].median().reset_index()
              .rename(columns={'well_lat':'proxy_lat','well_lon':'proxy_lon'}))
candidates = candidates.merge(well_proxy, on='planSource', how='left')
candidates['coord_lat'] = candidates['Latitude'].fillna(candidates['proxy_lat'])
candidates['coord_lon'] = candidates['Longitude'].fillna(candidates['proxy_lon'])
candidates = candidates[candidates['coord_lat'].notna()].reset_index(drop=True)
print(f'Candidate sources with coordinates: {len(candidates):,}')

## Name extraction and normalization

In [ ]:
WATER_TERMS = (
    r'\b(?:creek|river|run|stream|lake|pond|reservoir|brook|branch|'
    r'fork|hollow|dam|hatchery|spring|susquehanna|wyalusing)\b'
)

_ABBREVS = [
    (r'\bN\.?\s+',  'North '),  (r'\bS\.?\s+',  'South '),
    (r'\bE\.?\s+',  'East '),   (r'\bW\.?\s+',  'West '),
    (r'\bNE\.?\s+', 'Northeast '), (r'\bNW\.?\s+', 'Northwest '),
    (r'\bSE\.?\s+', 'Southeast '), (r'\bSW\.?\s+', 'Southwest '),
    (r'\bBr\.?\s+', 'Branch '), (r'\bFk\.?\s+', 'Fork '),
    (r'\bCk\.?\b',  'Creek'),   (r'\bCr\.?\b',  'Creek'),
    (r'\bUnt\.?\b', 'Unnamed Tributary'), (r'\bUNT\.?\b', 'Unnamed Tributary'),
    (r'\bMon\b',    'Monongahela'),
]

def normalize_name(s):
    if not s or pd.isna(s): return ''
    s = str(s)
    for pat, repl in _ABBREVS:
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)
    s = re.sub(r'[^a-z\s]', ' ', s.lower())
    return re.sub(r'\s+', ' ', s).strip()

_OP_PREFIXES = {
    'cabot','coterra','bkv','sfgs','sgfs','bnr','carrizo','repsol',
    'chesapeake','eqt','range','swn','talisman','cnx','well','field',
    'diaz','garrison',
}

_DELIM = re.compile(r',|\s+-\s+|\s+@\s+')

def extract_search_name(planSource):
    if pd.isna(planSource): return ''
    s = str(planSource)
    s = re.sub(r'[\[\(]SRBC[^\]\)]*[\]\)]', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\bSRBC\s+Docket\s+(?:No\.|Number)?\s*\d[\d\-]*', '', s, flags=re.IGNORECASE)
    s_np = re.sub(r'\([^)]*\)', '', s)
    s_np = re.sub(r'\b(WV|PA|OH|NY|MD|NJ)\b', '', s_np)
    s_np = re.sub(r'\b(source|station|rhl|wwp|wwd|dcnr|swpa)\b.*$', '', s_np, flags=re.IGNORECASE)
    s_np = s_np.strip(' ,.-')

    feature = None
    for seg in reversed([seg.strip() for seg in _DELIM.split(s_np)]):
        if re.search(WATER_TERMS, seg, re.IGNORECASE):
            feature = seg.strip(' -.')
            break
    if feature is None:
        for paren in re.findall(r'\(([^)]+)\)', s):
            if re.search(WATER_TERMS, paren, re.IGNORECASE):
                feature = paren.strip()
                break
    if feature is None and re.search(WATER_TERMS, s_np, re.IGNORECASE):
        feature = s_np
    if feature is None:
        return ''

    m_last = None
    for m in re.finditer(WATER_TERMS, feature, re.IGNORECASE):
        m_last = m
    if m_last:
        feature = feature[:m_last.end()].strip(' -.')

    words = feature.split()
    while words:
        tok = re.sub(r'[\-\.\#\d]', '', words[0]).lower()
        if tok in _OP_PREFIXES or tok == '':
            words = words[1:]
        else:
            break
    return ' '.join(words).strip()


candidates['search_name']      = candidates['planSource'].apply(extract_search_name)
candidates['search_name_norm'] = candidates['search_name'].apply(normalize_name)
nhd['gnis_norm'] = nhd['gnis_name'].apply(normalize_name)
nhd['cx'] = nhd.geometry.centroid.x
nhd['cy'] = nhd.geometry.centroid.y

matched_cands = candidates[candidates['search_name_norm'] != ''].copy()
print(f'Candidates with extractable name: {len(matched_cands):,} / {len(candidates):,}')
print()
checks = matched_cands[matched_cands['planSource'].str.contains(
    'Susquehanna Gas|Rock Run|Columbia Unt', case=False, na=False)]
print(checks[['planSource','search_name']].to_string())

## Spatial + fuzzy matcher

In [ ]:
_DEG_LAT = 1 / 111.0
_DEG_LON = 1 / 85.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))


def find_nhd_match(search_norm, lat, lon, radius_km=50, top_n=3):
    if not search_norm:
        return []
    dlat, dlon = radius_km*_DEG_LAT, radius_km*_DEG_LON
    nearby = nhd[
        (nhd['gnis_norm'] != '') &
        nhd['cy'].between(lat-dlat, lat+dlat) &
        nhd['cx'].between(lon-dlon, lon+dlon)
    ]
    if nearby.empty:
        return []
    scores = nearby['gnis_norm'].apply(lambda n: fuzz.token_sort_ratio(search_norm, n))
    top_idx = scores.nlargest(top_n).index
    results = []
    for idx in top_idx:
        row = nhd.loc[idx]
        dist = haversine_km(lat, lon, row['cy'], row['cx'])
        results.append({
            'nhd_id':    row['permanent_identifier'],
            'nhd_name':  row['gnis_name'],
            'nhd_ftype': row['ftype'],
            'nhd_layer': row['layer'],
            'score':     scores[idx],
            'dist_km':   round(dist, 1),
        })
    return sorted(results, key=lambda r: (-r['score'], r['dist_km']))


print(f'Running matcher on {len(matched_cands):,} candidates...')
records = []
for _, row in matched_cands.iterrows():
    hits = find_nhd_match(row['search_name_norm'], row['coord_lat'], row['coord_lon'])
    best = hits[0] if hits else {}
    records.append({
        'planSource':  row['planSource'],
        'source_type': row['source_type'],
        'has_srbc':    row['has_srbc'],
        'search_name': row['search_name'],
        'coord_lat':   row['coord_lat'],
        'coord_lon':   row['coord_lon'],
        'nhd_id':      best.get('nhd_id'),
        'nhd_name':    best.get('nhd_name'),
        'nhd_ftype':   best.get('nhd_ftype'),
        'nhd_layer':   best.get('nhd_layer'),
        'score':       best.get('score'),
        'dist_km':     best.get('dist_km'),
    })

results = pd.DataFrame(records)
print('Done.')
print(f'Matched (score >= 80): {(results["score"] >= 80).sum():,}')
print(f'Matched (score >= 60): {(results["score"] >= 60).sum():,}')
print(f'No NHD feature nearby: {results["nhd_name"].isna().sum():,}')

## Results review

In [ ]:
bins   = [0, 50, 60, 70, 80, 90, 100]
labels = ['<50','50-59','60-69','70-79','80-89','90-100']
results['score_bin'] = pd.cut(results['score'], bins=bins, labels=labels, right=True)
print('Score distribution:')
print(results['score_bin'].value_counts().sort_index().to_string())
print()
print('Median distance by score bin (km):')
print(results.groupby('score_bin', observed=True)['dist_km'].median().to_string())

In [ ]:
high = results[results['score'] >= 80].sort_values('score', ascending=False)
print(f'High-confidence matches (score >= 80): {len(high)}')
print(high[['search_name','nhd_name','score','dist_km','nhd_ftype','nhd_layer']]
      .head(30).to_string())

In [ ]:
low = results[results['score'].isna() | (results['score'] < 60)]
print(f'Low/no match (score < 60 or no nearby feature): {len(low)}')
print(low[['search_name','nhd_name','score','dist_km']].head(30).to_string())

In [ ]:
results.to_parquet('data/nhd_match_results.parquet', index=False)
print('Saved: data/nhd_match_results.parquet')

In [ ]:
## Re-match SRBC rows with precise coordinates and source names
#
# For SRBC-tagged sources we have two improvements over the initial pass:
#   1. Precise withdrawal-point coordinates from the docket PDFs (vs. well proxy)
#   2. SRBC-confirmed source name (e.g. "Susquehanna River" vs. "Susquehanna")
#
# Strategy: try three (name, coord) combinations per row; keep the best score.
#   Run A: planSource-extracted name  + SRBC coords
#   Run B: SRBC approved_source name + SRBC coords
#   Run C: planSource-extracted name  + original coords  ← fallback if SRBC coords mislead

srbc_coords = pd.read_parquet('data/srbc_coords_lookup.parquet')

srbc_cands = matched_cands[matched_cands['has_srbc']].copy()
srbc_cands = srbc_cands.merge(
    srbc_coords[['planSource', 'srbc_lat', 'srbc_lon', 'srbc_source_name']],
    on='planSource', how='left'
)
srbc_cands['use_lat'] = srbc_cands['srbc_lat'].fillna(srbc_cands['coord_lat'])
srbc_cands['use_lon'] = srbc_cands['srbc_lon'].fillna(srbc_cands['coord_lon'])
srbc_cands['srbc_name_norm'] = srbc_cands['srbc_source_name'].apply(normalize_name)

print(f'SRBC candidates: {len(srbc_cands)}'
      f'  (with SRBC coords: {srbc_cands["srbc_lat"].notna().sum()},'
      f'  SRBC name: {srbc_cands["srbc_name_norm"].ne("").sum()})')

srbc_records = []
for _, row in srbc_cands.iterrows():
    lat_s, lon_s   = row['use_lat'], row['use_lon']
    lat_o, lon_o   = row['coord_lat'], row['coord_lon']
    plan_norm      = row['search_name_norm']
    srbc_norm      = row['srbc_name_norm']

    runs = [
        (plan_norm, lat_s, lon_s, row['search_name']),
        (srbc_norm, lat_s, lon_s, row['srbc_source_name']),
        (plan_norm, lat_o, lon_o, row['search_name']),
    ]

    best, best_label = {}, row['search_name']
    for norm, lat, lon, label in runs:
        if not norm or pd.isna(lat):
            continue
        hits = find_nhd_match(norm, lat, lon)
        if hits and (hits[0].get('score', 0) or 0) > (best.get('score', 0) or 0):
            best = hits[0]
            best_label = label

    srbc_records.append({
        'planSource':  row['planSource'],
        'source_type': row['source_type'],
        'has_srbc':    True,
        'search_name': best_label,
        'coord_lat':   lat_s,
        'coord_lon':   lon_s,
        'nhd_id':      best.get('nhd_id'),
        'nhd_name':    best.get('nhd_name'),
        'nhd_ftype':   best.get('nhd_ftype'),
        'nhd_layer':   best.get('nhd_layer'),
        'score':       best.get('score'),
        'dist_km':     best.get('dist_km'),
    })

srbc_results = pd.DataFrame(srbc_records)

# Before/after comparison
old_scores = results[results['has_srbc']].set_index('planSource')['score']
new_scores = srbc_results.set_index('planSource')['score']
comp = pd.DataFrame({'old': old_scores, 'new': new_scores}).dropna()
comp['delta'] = comp['new'] - comp['old']
print(f'\nSRBC re-match vs. original:')
print(f'  Improved (delta > 0):  {(comp.delta > 0).sum()}')
print(f'  No change:             {(comp.delta == 0).sum()}')
print(f'  Worsened (delta < 0):  {(comp.delta < 0).sum()}')
print(f'  Mean delta:            {comp.delta.mean():.1f}')
print(f'\n  Score >= 90 (new): {(srbc_results["score"] >= 90).sum()}')
print(f'  Score >= 80 (new): {(srbc_results["score"] >= 80).sum()}')

In [ ]:
# Replace SRBC rows in the results table with the improved re-match
results_updated = pd.concat([
    results[~results['has_srbc']],
    srbc_results,
], ignore_index=True)

bins   = [0, 50, 60, 70, 80, 90, 100]
labels = ['<50', '50-59', '60-69', '70-79', '80-89', '90-100']
results_updated['score_bin'] = pd.cut(results_updated['score'], bins=bins, labels=labels, right=True)
print('Score distribution after SRBC upgrade:')
print(results_updated['score_bin'].value_counts().sort_index().to_string())

results_updated.drop(columns='score_bin').to_parquet('data/nhd_match_results.parquet', index=False)
print('\nSaved updated: data/nhd_match_results.parquet')

In [ ]:
## WV border re-match with combined PA+WV NHD and enhanced name expansions
# Re-match candidates near the PA/WV border using:
#   1. Combined PA+WV NHD (adds Monongahela, Dunkard Creek, Fish Creek WV, etc.)
#   2. WV-specific name expansions (Mon → Monongahela, Nunkard → Dunkard, etc.)
#   3. Fallback for generic extracted names ("River", "Creek" alone) — derive from planSource

_WV_ABBREVS = [
    (r'\bMon\b',     'Monongahela'),
    (r'\bNunkard\b', 'Dunkard'),     # common OCR/typo variant of Dunkard
    (r'\bWhg\b',     'Wheeling'),
]

_GENERIC_NAMES = {'river', 'creek', 'run', 'stream', 'lake', 'pond',
                  'branch', 'fork', 'brook', 'hollow', 'dam'}


def normalize_wv(s):
    """normalize_name + WV-specific abbreviation expansions."""
    if not s or pd.isna(s): return ''
    s = str(s)
    for pat, repl in _ABBREVS:
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)
    for pat, repl in _WV_ABBREVS:
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)
    s = re.sub(r'[^a-z\s]', ' ', s.lower())
    return re.sub(r'\s+', ' ', s).strip()


def get_wv_search_norm(row):
    """Return the best WV-normalized search string for a border candidate.

    When extract_search_name() yields only a bare water-type word (e.g. 'River'
    from 'Monongahela @ River Speers'), fall back to the planSource stripped of
    @ / , / - location suffixes, then apply WV normalization.
    """
    sn = (row['search_name'] or '').strip()
    if not sn or sn.lower() in _GENERIC_NAMES or len(sn.replace(' ', '')) < 4:
        ps = str(row['planSource'])
        ps = re.sub(r'[\[\(]SRBC[^\]\)]*[\]\)]', '', ps, flags=re.IGNORECASE)
        ps = re.sub(r'\([^)]*\)', '', ps)
        ps = re.sub(r'\s+@\s+.*$', '', ps)
        ps = re.sub(r',.*$', '', ps)
        ps = ps.strip(' ,.-')
        # Trim at last water term
        m_last = None
        for m in re.finditer(WATER_TERMS, ps, re.IGNORECASE):
            m_last = m
        if m_last:
            ps = ps[:m_last.end()].strip()
        return normalize_wv(ps) if ps else ''
    return normalize_wv(sn)


# Load combined PA+WV NHD
print('Loading combined PA+WV NHD...')
nhd_cmb = gpd.read_file('data/NHD_combined_named.gpkg')
nhd_cmb['gnis_norm'] = nhd_cmb['gnis_name'].apply(normalize_wv)  # use WV norms for index too
nhd_cmb['cx'] = nhd_cmb.geometry.centroid.x
nhd_cmb['cy'] = nhd_cmb.geometry.centroid.y
print(f'  Combined features: {len(nhd_cmb):,}')


def find_nhd_match_cmb(search_norm, lat, lon, radius_km=50, top_n=3):
    if not search_norm:
        return []
    dlat, dlon = radius_km * _DEG_LAT, radius_km * _DEG_LON
    nearby = nhd_cmb[
        (nhd_cmb['gnis_norm'] != '') &
        nhd_cmb['cy'].between(lat - dlat, lat + dlat) &
        nhd_cmb['cx'].between(lon - dlon, lon + dlon)
    ]
    if nearby.empty:
        return []
    scores = nearby['gnis_norm'].apply(lambda n: fuzz.token_sort_ratio(search_norm, n))
    top_idx = scores.nlargest(top_n).index
    out = []
    for idx in top_idx:
        r = nhd_cmb.loc[idx]
        dist = haversine_km(lat, lon, r['cy'], r['cx'])
        out.append({
            'nhd_id':    r['permanent_identifier'],
            'nhd_name':  r['gnis_name'],
            'nhd_ftype': r['ftype'],
            'nhd_layer': r['layer'],
            'score':     scores[idx],
            'dist_km':   round(dist, 1),
        })
    return sorted(out, key=lambda x: (-x['score'], x['dist_km']))


# Target: candidates west of -79.5 lon (PA/WV border zone)
WV_LON_THRESH = -79.5

wv_border = matched_cands[matched_cands['coord_lon'] < WV_LON_THRESH].copy()
wv_border['wv_norm'] = wv_border.apply(get_wv_search_norm, axis=1)
cur_score_map = results_updated.set_index('planSource')['score']
wv_border['cur_score'] = wv_border['planSource'].map(cur_score_map).fillna(0)

print(f'Western border candidates: {len(wv_border):,}  (coord_lon < {WV_LON_THRESH})')
print()
print(wv_border[['planSource', 'search_name', 'wv_norm', 'cur_score']].to_string())

improved_rows = []
for _, row in wv_border.iterrows():
    wv_n = row['wv_norm']
    lat, lon = row['coord_lat'], row['coord_lon']
    cur = row['cur_score'] or 0
    hits = find_nhd_match_cmb(wv_n, lat, lon)
    if hits and hits[0]['score'] > cur:
        best = hits[0]
        improved_rows.append({
            'planSource':  row['planSource'],
            'source_type': row['source_type'],
            'has_srbc':    row['has_srbc'],
            'search_name': row['search_name'],
            'coord_lat':   lat,
            'coord_lon':   lon,
            'nhd_id':      best['nhd_id'],
            'nhd_name':    best['nhd_name'],
            'nhd_ftype':   best['nhd_ftype'],
            'nhd_layer':   best['nhd_layer'],
            'score':       best['score'],
            'dist_km':     best['dist_km'],
        })

print(f'\nWV re-match improved: {len(improved_rows)} / {len(wv_border)} border candidates')
if improved_rows:
    wv_imp = pd.DataFrame(improved_rows)
    print(wv_imp[['planSource', 'search_name', 'score', 'nhd_name', 'dist_km']].to_string())

    improved_ps = {r['planSource'] for r in improved_rows}
    results_updated = pd.concat([
        results_updated[~results_updated['planSource'].isin(improved_ps)],
        wv_imp,
    ], ignore_index=True)

    bins   = [0, 50, 60, 70, 80, 90, 100]
    labels = ['<50', '50-59', '60-69', '70-79', '80-89', '90-100']
    tmp = results_updated.copy()
    tmp['score_bin'] = pd.cut(tmp['score'], bins=bins, labels=labels, right=True)
    print('\nUpdated score distribution after WV re-match:')
    print(tmp['score_bin'].value_counts().sort_index().to_string())

    results_updated.to_parquet('data/nhd_match_results.parquet', index=False)
    print('\nSaved updated: data/nhd_match_results.parquet')

In [ ]:
jct_matched = jct.merge(
    results_updated[['planSource', 'search_name', 'nhd_id', 'nhd_name',
                     'nhd_ftype', 'nhd_layer', 'score', 'dist_km']],
    on='planSource', how='left'
)

def confidence_tier(score):
    if pd.isna(score): return 'unmatched'
    if score >= 90:    return 'high (>=90)'
    if score >= 80:    return 'good (80-89)'
    if score >= 60:    return 'fair (60-79)'
    return                    'low (<60)'

jct_matched['match_tier'] = jct_matched['score'].apply(confidence_tier)
tier_order = ['high (>=90)', 'good (80-89)', 'fair (60-79)', 'low (<60)', 'unmatched']

def vol_table(df):
    return (
        df.groupby('match_tier')['volume']
        .agg(records='count', volume_gal='sum')
        .assign(
            volume_Mgal = lambda d: (d['volume_gal'] / 1e6).round(1),
            pct_volume  = lambda d: (d['volume_gal'] / d['volume_gal'].sum() * 100).round(1),
        )
        .reindex(tier_order)
        [['records', 'volume_Mgal', 'pct_volume']]
    )

print('=== All junction rows ===')
print(vol_table(jct_matched).to_string())
print()
cand_rows = jct_matched[jct_matched['source_type'].isin(['surface_direct', 'impoundment', 'srbc_only'])]
print('=== Surface / impoundment / SRBC candidates only ===')
print(vol_table(cand_rows).to_string())

jct_matched.to_parquet('data/junction_nhd_matched.parquet', index=False)
print(f'\nSaved: data/junction_nhd_matched.parquet  ({len(jct_matched):,} rows, {len(jct_matched.columns)} columns)')

## Join match results back to junction table
Attach NHD match to every junction-table row to report volume by match confidence.

## Pass 4 — DEP-assisted NHD re-match

Targets two groups missed by the original three passes:
1. **New candidates**: sources reclassified from `dont_know`/`ambiguous` → `surface_direct`/`impoundment` by `dep_matcher.ipynb` — never sent through the NHD matcher
2. **Re-match fair/low**: existing score 60–79 matches that now have precise DEP withdrawal-point coordinates (previously used well-proxy coords)

Search name strategy (best available, in order):
- `extract_search_name(planSource)` — primary
- `extract_search_name(dep_src)` — fallback for SWW/WI entries (e.g. `"SUSQUEHANNA RIVER - SALSMAN"`)
- SRBC confirmed source name — for SRBC-tagged sources

Uses combined PA+WV NHD throughout.

In [ ]:
# Reload combined NHD if not already loaded (supports running Pass 4 standalone)
try:
    _ = nhd_cmb
    print(f'nhd_cmb already loaded ({len(nhd_cmb):,} features)')
except NameError:
    print('Loading combined PA+WV NHD...')
    nhd_cmb = gpd.read_file('data/NHD_combined_named.gpkg')
    nhd_cmb['gnis_norm'] = nhd_cmb['gnis_name'].apply(normalize_wv)
    nhd_cmb['cx'] = nhd_cmb.geometry.centroid.x
    nhd_cmb['cy'] = nhd_cmb.geometry.centroid.y
    print(f'  {len(nhd_cmb):,} features loaded')

# Load post-DEP junction table
jct_dep = pd.read_parquet('data/junction_dep_updated.parquet')

# SRBC coords and well proxy
srbc_info = pd.read_parquet('data/srbc_coords_lookup.parquet')
skinny_mini = pd.read_parquet(SKINNY_PATH, columns=['api10', 'bgLatitude', 'bgLongitude']).dropna()
well_proxy_p4 = (
    jct_dep[['planSource', 'api10']].drop_duplicates()
    .merge(skinny_mini, on='api10', how='left')
    .groupby('planSource')[['bgLatitude', 'bgLongitude']].median()
    .rename(columns={'bgLatitude': 'well_lat', 'bgLongitude': 'well_lon'})
    .reset_index()
)

per_src = (
    jct_dep.groupby('planSource').agg(
        source_type=('source_type', 'first'),
        match_tier=('match_tier', 'first'),
        dep_lat=('dep_lat', 'first'),
        dep_lon=('dep_lon', 'first'),
        dep_src=('dep_src', 'first'),
        dep_score=('dep_score', 'first'),
    ).reset_index()
    .merge(srbc_info[['planSource', 'srbc_lat', 'srbc_lon', 'srbc_source_name']],
           on='planSource', how='left')
    .merge(well_proxy_p4, on='planSource', how='left')
)
per_src['has_srbc_tag'] = per_src['planSource'].str.contains(
    r'srbc\s+docket', case=False, na=False)

def pick_coord(row):
    if pd.notna(row['dep_lat']):  return row['dep_lat'],  row['dep_lon'],  'dep'
    if pd.notna(row['srbc_lat']): return row['srbc_lat'], row['srbc_lon'], 'srbc'
    if pd.notna(row['well_lat']): return row['well_lat'], row['well_lon'], 'well'
    return np.nan, np.nan, 'none'

coords = per_src.apply(pick_coord, axis=1, result_type='expand')
per_src[['coord_lat', 'coord_lon', 'coord_src']] = coords.values

tgt_new = per_src[
    per_src['source_type'].isin(['surface_direct', 'impoundment']) &
    (per_src['match_tier'] == 'unmatched') &
    per_src['coord_lat'].notna()
].copy()

tgt_rematch = per_src[
    per_src['source_type'].isin(['surface_direct', 'impoundment', 'srbc_only']) &
    per_src['match_tier'].isin(['fair (60-79)', 'low (<60)']) &
    per_src['dep_lat'].notna()
].copy()

targets = pd.concat([tgt_new, tgt_rematch], ignore_index=True).drop_duplicates('planSource')
print(f'Pass 4 targets:')
print(f'  New unmatched surface/impoundment with coord: {len(tgt_new):,}')
print(f'  Existing fair/low with DEP coord:              {len(tgt_rematch):,}')
print(f'  Total unique:                                  {len(targets):,}')


In [ ]:
def get_search_attempts(row):
    attempts = []
    ps_name = extract_search_name(str(row['planSource']))
    ps_norm = normalize_wv(ps_name)
    if ps_norm:
        attempts.append((ps_norm, row['coord_lat'], row['coord_lon'], ps_name))
    if pd.notna(row.get('dep_src')) and (row.get('dep_score') or 0) >= 80:
        dep_name = extract_search_name(str(row['dep_src']))
        dep_norm = normalize_wv(dep_name)
        if dep_norm and dep_norm != ps_norm:
            attempts.append((dep_norm, row['coord_lat'], row['coord_lon'], row['dep_src']))
    if row.get('has_srbc_tag') and pd.notna(row.get('srbc_source_name')):
        srbc_norm = normalize_wv(str(row['srbc_source_name']))
        s_lat = row['srbc_lat'] if pd.notna(row.get('srbc_lat')) else row['coord_lat']
        s_lon = row['srbc_lon'] if pd.notna(row.get('srbc_lon')) else row['coord_lon']
        if srbc_norm and pd.notna(s_lat):
            attempts.append((srbc_norm, s_lat, s_lon, row['srbc_source_name']))
    return attempts

print(f'Running Pass 4 on {len(targets):,} targets...')
p4_records = []
for _, row in targets.iterrows():
    attempts = get_search_attempts(row)
    best, best_label = {}, str(row['planSource'])
    for norm, lat, lon, label in attempts:
        if not norm or pd.isna(lat):
            continue
        hits = find_nhd_match_cmb(norm, lat, lon)
        if hits and hits[0].get('score', 0) > best.get('score', 0):
            best = hits[0]
            best_label = label
    if best:
        p4_records.append({
            'planSource':  row['planSource'],
            'search_name': best_label,
            'nhd_id':      best['nhd_id'],
            'nhd_name':    best['nhd_name'],
            'nhd_ftype':   best['nhd_ftype'],
            'nhd_layer':   best['nhd_layer'],
            'score':       best['score'],
            'dist_km':     best['dist_km'],
        })

p4_df = pd.DataFrame(p4_records)
print(f'Done. {len(p4_df):,} sources returned a match.')
print(f'  Score >= 90: {(p4_df["score"] >= 90).sum():,}')
print(f'  Score >= 80: {(p4_df["score"] >= 80).sum():,}')
print(f'  Score >= 60: {(p4_df["score"] >= 60).sum():,}')
print(f'  Score <  60: {(p4_df["score"] < 60).sum():,}')
print()
print('Top 30 Pass 4 matches by score:')
print(p4_df[['planSource', 'search_name', 'nhd_name', 'score', 'dist_km']]
      .sort_values('score', ascending=False).head(30).to_string(index=False))


In [ ]:
cur_match  = pd.read_parquet('data/nhd_match_results.parquet')
cur_score  = cur_match.set_index('planSource')['score'].to_dict()
rematch_ps = set(tgt_rematch['planSource'])

apply_rows = []
for _, r in p4_df.iterrows():
    ps = r['planSource']
    if ps in rematch_ps:
        if r['score'] > cur_score.get(ps, 0):
            apply_rows.append(r)
    else:
        if r['score'] >= 60:
            apply_rows.append(r)

apply_df = pd.DataFrame(apply_rows)
print(f'Applying {len(apply_df):,} Pass 4 matches:')
print(f'  New sources matched (score >= 60): '
      f'{len(apply_df[~apply_df["planSource"].isin(rematch_ps)]):,}')
print(f'  Existing matches improved:         '
      f'{len(apply_df[apply_df["planSource"].isin(rematch_ps)]):,}')
print(f'  Score >= 90: {(apply_df["score"] >= 90).sum():,}')
print(f'  Score >= 80: {(apply_df["score"] >= 80).sum():,}')

apply_df['match_tier'] = apply_df['score'].apply(confidence_tier)
apply_map = apply_df.set_index('planSource')
update_ps  = set(apply_map.index)

mask = jct_dep['planSource'].isin(update_ps)
for col in ['search_name', 'nhd_id', 'nhd_name', 'nhd_ftype', 'nhd_layer',
            'score', 'dist_km', 'match_tier']:
    jct_dep.loc[mask, col] = jct_dep.loc[mask, 'planSource'].map(apply_map[col])

jct_dep.to_parquet('data/junction_dep_updated.parquet', index=False)
print(f'\nSaved: data/junction_dep_updated.parquet')

new_rows = apply_df[['planSource', 'search_name', 'nhd_id', 'nhd_name',
                      'nhd_ftype', 'nhd_layer', 'score', 'dist_km']]
cur_match_upd = pd.concat(
    [cur_match[~cur_match['planSource'].isin(update_ps)], new_rows],
    ignore_index=True
)
cur_match_upd.to_parquet('data/nhd_match_results.parquet', index=False)
print(f'Saved: data/nhd_match_results.parquet')

TIER_ORDER = ['high (>=90)', 'good (80-89)', 'fair (60-79)', 'low (<60)', 'unmatched']
def vtbl(df, label=''):
    t = (df.groupby('match_tier')['volume']
         .agg(records='count',
              volume_Mgal=lambda x: round(x.sum()/1e6, 1),
              pct=lambda x: round(x.sum()/df['volume'].sum()*100, 1))
         .reindex(TIER_ORDER))
    if label: print(f'\n=== {label} ===')
    print(t.to_string())

vtbl(jct_dep, 'All junction rows — after Pass 4')
vtbl(jct_dep[jct_dep['source_type'].isin(['surface_direct','impoundment','srbc_only'])],
     'Surface/impoundment/SRBC candidates only')
